In [1]:
import pandas as pd
import numpy as np

from scipy.stats import wilcoxon
from collections import defaultdict
from default_vars import PAIRED_MAIN_STATEMENTS, UNCERTAINTY_EXPRESSIONS

TRUE_MAIN_STATEMENTS, FALSE_MAIN_STATEMENTS = zip(*PAIRED_MAIN_STATEMENTS) 

In this notebook, we will perform the analysis by the truth-falsity of the evaluated statement.

In [2]:
all_models_top_preds = pd.read_csv("../../results/greedy/canonic_data.csv", index_col=0)
all_models_top_preds = all_models_top_preds[all_models_top_preds["statement_type"].apply(lambda x: x.lower().startswith("verifiable"))]
all_models_top_preds = all_models_top_preds[all_models_top_preds["uncertainty_expression"].isin(UNCERTAINTY_EXPRESSIONS)]
all_models_top_preds["__model"] = all_models_top_preds["model"] 
all_models_top_preds["model"] = all_models_top_preds["model"] + "__" + all_models_top_preds["__methodology"]

# Main experiment
all_models_top_preds_main = all_models_top_preds[all_models_top_preds["__dataset"] == 'main']
# Generalization experiment
all_models_top_preds_ai2arc = all_models_top_preds[all_models_top_preds["__dataset"] == 'ai2arc']

# Number of predictions per model
all_models_top_preds_main.groupby(["model"]).count() 
# all_models_top_preds_main.groupby(["__model", "__methodology"]).count() 

,uncertainty_expression,numerical_response,speaker_name,speaker_gender,template,statement_id,statement_type,statement_uuid,statement,__orig_statement_type,__results_filepath,__basename,__n_shots,__dataset,__methodology,__model
model,,,,,,,,,,,,,,,,
allenai/OLMo-7B-Instruct__full-prob-argmax,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780
allenai/OLMo-7B-Instruct__sampling-based,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780
google/gemma-1.1-2b-it__full-prob-argmax,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780
gpt-3.5-turbo-0125__top-k,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780
gpt-4-turbo-2024-04-09__top-k,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780
gpt-4o-2024-05-13__top-k,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780
lmsys/vicuna-13b-v1.5__full-prob-argmax,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780
meta-llama/Llama-3-70b-chat-hf__sampling-based,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780
meta-llama/Meta-Llama-3-8B-Instruct__full-prob-argmax,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780,780


In [3]:
all_models_top_preds_ai2arc.groupby("model").count()

,uncertainty_expression,numerical_response,speaker_name,speaker_gender,template,statement_id,statement_type,statement_uuid,statement,__orig_statement_type,__results_filepath,__basename,__n_shots,__dataset,__methodology,__model
model,,,,,,,,,,,,,,,,
allenai/OLMo-7B-Instruct__sampling-based,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278
google/gemma-1.1-2b-it__sampling-based,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278
gpt-3.5-turbo-0125__top-k,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278
gpt-4-turbo-2024-04-09__top-k,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278
gpt-4o-2024-05-13__top-k,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278
meta-llama/Llama-3-70b-chat-hf__sampling-based,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278
mistralai/Mixtral-8x22B-Instruct-v0.1__sampling-based,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278
models/gemini-pro__sampling-based,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278,5278


## Paired T-test 

We'd like to measure whether the predictions for the true statements and false statements are significant at a pair-level. This is a necessary test to ensure that there are clear distinctions between the results when we minimally change the statements from true to false. 

- 'Big Ben is located in London.' vs  'Big Ben is located in Berlin.'
- 'Great Britain directly borders 0 countries.' vs. 'Great Britain directly borders 2 countries.'
- 'the Colosseum, a famous landmark in Rome, was originally built as an Amphitheatre.' vs 'the Colosseum, a famous landmark in Rome, was originally built - as a Cathedral.'
- 'the US president lives in the White House.' vs 'the UK Prime Minister lives in the White House.'
- 'Mount Everest is part of the Himalayas mountain range.' vs 'Mount Everest is part of the Andes mountain range.'
- 'the Eiffel Tower is made of iron.' vs  'the Eiffel Tower is made of aluminum.'



In [4]:
def methodology_fmt(model):
    if "full-prob-argmax" in model:
        return "full"
    elif "top-k" in model:
        return "top-k"
    else:
        return "sampling"


def model_fmt(m):
    if "gemma" in m:
        return "\\gemma"
    elif "olmo" in m.lower():
        return "\\olmo"
    elif "llama" in m.lower() and "8" in m:
        return "\\llamasmall"
    elif "llama" in m.lower() and "8" not in m:
        return "\\llama"
    elif "gemini" in m:
        return "\\gemini"
    elif "Mixtral-8x22B" in m:
        return "\\mixtralmoelg"
    elif "gpt-4o" in m:
        return "\\gptfo"
    elif "gpt-4-turbo" in m:
        return "\\gptf"
    elif "gpt-3.5" in m:
        return "\\chatgpt"



def apply_wilcoxon(df, expr, unc_col):
    df_expr = df[df[unc_col] == expr].copy()
    return wilcoxon(df_expr["true_statement_response"], y=df_expr["false_statement_response"], alternative="greater")


def run_paired_rest(data, models, paired_sents, model_col="model", unc_col="uncertainty_expression", uncertainty_expressions=UNCERTAINTY_EXPRESSIONS):
    
    all_results =  defaultdict(list)
    all_model_results = {}
    for model in models:
        print("Processing", model)
        main_results = defaultdict(list)
        model_df = data[data[model_col] == model]
        model_df = model_df.sort_values(unc_col)
    
        for i, (st_true, st_false) in enumerate(paired_sents):
            model_df_true = model_df[model_df["statement"] == st_true]
            model_df_false = model_df[model_df["statement"] == st_false]
            assert len(model_df_true) == len(model_df_false) == len(uncertainty_expressions)
            assert all(model_df_true["uncertainty_expression"].values == model_df_false["uncertainty_expression"].values)
    
            main_results["model"].extend([model] * len(uncertainty_expressions))
            main_results["pair_id"].extend([i] * len(uncertainty_expressions))
            main_results["uncertainty_expression"].extend(model_df_true["uncertainty_expression"].values.tolist())
            main_results["true_statement_response"].extend(model_df_true["numerical_response"].values.tolist())
            main_results["false_statement_response"].extend(model_df_false["numerical_response"].values.tolist())
            
        main_results = pd.DataFrame(main_results)
        assert not (pd.DataFrame(main_results)==-1).any().any()
        stats = wilcoxon(main_results["true_statement_response"], y=main_results["false_statement_response"], alternative="greater")
        all_model_results[model] = {"Statistic": stats.statistic, "p-value": stats.pvalue}
    
        for expr in uncertainty_expressions:
            try:
                values = apply_wilcoxon(main_results, expr, "uncertainty_expression")
                stat, pval = values.statistic, values.pvalue
                all_results["wilcoxon statistic"].append(values.statistic)
                all_results["wilcoxon pval"].append(values.pvalue)
                all_results["wilcoxon"].append("$" + str({values.statistic}) + "_{" + f"{values.pvalue:.2e}" +"}$")
            except:
                print("-> Skipped expression", expr)
                all_results["wilcoxon statistic"].append(None)
                all_results["wilcoxon pval"].append(None)
                all_results["wilcoxon"].append(None)
                
            all_results["model"].append(model)
            all_results["expr"].append(expr)

    all_model_results_df = pd.DataFrame(all_model_results).T.reset_index()
    all_model_results_df["Statistic"] = all_model_results_df["Statistic"].apply(lambda x: f"{x:.2f}")

    all_model_results_df["p-value"] = all_model_results_df["p-value"].apply(
        lambda x: "\\textbf{<0.0001*}" if x < 0.0001 else f"{x:.4f}")
    
    
    all_model_results_df.insert(0, "methodology", all_model_results_df["index"].apply(methodology_fmt))
    all_model_results_df.insert(0, "model", all_model_results_df["index"].apply(model_fmt))
    return pd.DataFrame(all_results), all_model_results_df

### 1. Verifiable experiment
--- 

In [5]:
MAIN_MODELS = sorted(all_models_top_preds_main["model"].unique())

all_results, all_model_results_df = run_paired_rest(
    data=all_models_top_preds_main,
    models=MAIN_MODELS,
    paired_sents=PAIRED_MAIN_STATEMENTS,
)
all_results = pd.DataFrame(all_results)
all_results.pivot(index="expr", columns="model", values="wilcoxon")

Processing allenai/OLMo-7B-Instruct__full-prob-argmax
Processing allenai/OLMo-7B-Instruct__sampling-based
Processing google/gemma-1.1-2b-it__full-prob-argmax
Processing gpt-3.5-turbo-0125__top-k
Processing gpt-4-turbo-2024-04-09__top-k
-> Skipped expression almost certain
Processing gpt-4o-2024-05-13__top-k
Processing lmsys/vicuna-13b-v1.5__full-prob-argmax
-> Skipped expression uncertain
Processing meta-llama/Llama-3-70b-chat-hf__sampling-based
Processing meta-llama/Meta-Llama-3-8B-Instruct__full-prob-argmax
Processing mistralai/Mistral-7B-Instruct-v0.2__full-prob-argmax
Processing mistralai/Mixtral-8x22B-Instruct-v0.1__sampling-based
-> Skipped expression almost certain
-> Skipped expression very unlikely
-> Skipped expression highly unlikely
Processing models/gemini-pro__sampling-based


model,allenai/OLMo-7B-Instruct__full-prob-argmax,allenai/OLMo-7B-Instruct__sampling-based,google/gemma-1.1-2b-it__full-prob-argmax,gpt-3.5-turbo-0125__top-k,gpt-4-turbo-2024-04-09__top-k,gpt-4o-2024-05-13__top-k,lmsys/vicuna-13b-v1.5__full-prob-argmax,meta-llama/Llama-3-70b-chat-hf__sampling-based,meta-llama/Meta-Llama-3-8B-Instruct__full-prob-argmax,mistralai/Mistral-7B-Instruct-v0.2__full-prob-argmax,mistralai/Mixtral-8x22B-Instruct-v0.1__sampling-based,models/gemini-pro__sampling-based
expr,,,,,,,,,,,,
almost certain,${48.5}_{1.17e-02}$,${48.5}_{1.17e-02}$,${19.5}_{6.41e-01}$,${91.0}_{6.47e-04}$,None,${435.0}_{3.84e-07}$,${136.0}_{1.76e-04}$,${318.0}_{8.57e-06}$,${169.0}_{1.21e-04}$,${253.5}_{1.53e-04}$,None,${21.0}_{1.16e-02}$
doubtful,${1.0}_{1.59e-01}$,${1.0}_{1.59e-01}$,${6.5}_{8.01e-01}$,${123.5}_{1.96e-03}$,${351.0}_{2.68e-06}$,${206.5}_{6.33e-05}$,${1.0}_{1.59e-01}$,${215.0}_{1.91e-04}$,${11.0}_{4.58e-01}$,${158.0}_{5.87e-04}$,${246.5}_{3.24e-05}$,${127.5}_{2.33e-04}$
highly likely,${64.0}_{2.24e-03}$,${64.0}_{2.24e-03}$,${1.0}_{9.63e-01}$,${300.0}_{8.10e-06}$,${406.0}_{8.72e-07}$,${210.0}_{3.28e-05}$,${130.0}_{4.15e-04}$,${420.5}_{5.16e-06}$,${298.0}_{1.13e-05}$,${378.5}_{2.68e-05}$,${1.0}_{1.59e-01}$,${276.0}_{1.23e-05}$
highly unlikely,${35.5}_{2.06e-01}$,${35.5}_{2.06e-01}$,${1.5}_{5.00e-01}$,${89.5}_{1.21e-01}$,${210.0}_{3.87e-06}$,${210.0}_{2.33e-05}$,${45.0}_{2.94e-03}$,${104.0}_{2.25e-03}$,${26.0}_{3.38e-01}$,${156.5}_{8.27e-04}$,None,${55.0}_{7.83e-04}$
not likely,${43.5}_{1.74e-01}$,${43.5}_{1.74e-01}$,${6.0}_{8.41e-01}$,${226.5}_{5.03e-05}$,${351.0}_{3.32e-06}$,${105.0}_{3.18e-04}$,${3.0}_{7.86e-02}$,${183.0}_{1.29e-03}$,${3.5}_{7.10e-01}$,${183.0}_{1.54e-03}$,${372.5}_{3.94e-06}$,${135.0}_{6.08e-05}$
possible,${126.0}_{1.17e-03}$,${126.0}_{1.17e-03}$,${0.0}_{8.41e-01}$,${406.0}_{1.26e-06}$,${210.0}_{1.15e-05}$,${433.5}_{1.38e-06}$,${21.0}_{7.15e-03}$,${434.0}_{1.32e-06}$,${118.5}_{4.04e-04}$,${197.5}_{2.62e-04}$,${268.0}_{3.54e-04}$,${444.0}_{4.16e-07}$
probable,${54.0}_{2.52e-03}$,${54.0}_{2.52e-03}$,${15.0}_{6.70e-01}$,${351.0}_{2.76e-06}$,${378.0}_{2.28e-06}$,${406.0}_{1.78e-06}$,${151.5}_{1.39e-03}$,${337.0}_{1.94e-05}$,${253.0}_{1.84e-05}$,${251.0}_{2.78e-04}$,${378.0}_{2.70e-06}$,${406.0}_{1.81e-06}$
somewhat likely,${37.0}_{4.22e-02}$,${37.0}_{4.22e-02}$,${2.0}_{9.35e-01}$,${272.0}_{1.89e-05}$,${253.0}_{1.83e-05}$,${372.5}_{4.71e-06}$,${6.0}_{5.12e-02}$,${390.0}_{9.31e-06}$,${276.0}_{8.78e-06}$,${123.5}_{2.00e-03}$,${378.0}_{2.31e-06}$,${321.5}_{8.62e-06}$
somewhat unlikely,${9.0}_{3.42e-01}$,${9.0}_{3.42e-01}$,${3.0}_{7.75e-01}$,${171.0}_{4.88e-05}$,${325.0}_{6.55e-07}$,${351.0}_{3.46e-06}$,${20.0}_{1.28e-01}$,${153.0}_{5.29e-05}$,${149.0}_{1.46e-04}$,${66.0}_{1.34e-03}$,${231.0}_{2.24e-05}$,${144.5}_{1.73e-04}$


In [6]:
print(all_model_results_df.dropna().drop(["index"], axis=1).set_index("model").to_latex())

\begin{tabular}{llll}
\toprule
 & methodology & Statistic & p-value \\
model &  &  &  \\
\midrule
\olmo & full & 5309.50 & \textbf{<0.0001*} \\
\olmo & sampling & 5309.50 & \textbf{<0.0001*} \\
\gemma & full & 786.50 & 0.9979 \\
\chatgpt & top-k & 31180.00 & \textbf{<0.0001*} \\
\gptf & top-k & 34980.00 & \textbf{<0.0001*} \\
\gptfo & top-k & 36185.50 & \textbf{<0.0001*} \\
\llama & sampling & 34979.00 & \textbf{<0.0001*} \\
\llamasmall & full & 17115.00 & \textbf{<0.0001*} \\
\mixtralmoelg & sampling & 16051.00 & \textbf{<0.0001*} \\
\gemini & sampling & 25564.00 & \textbf{<0.0001*} \\
\bottomrule
\end{tabular}



### 2. Generalization experiment
----

In [7]:
def get_ai2arc_dataset_pairs(subset="easy") -> pd.DataFrame:
    if subset == "easy":
        gsheetkey="1ZdQRkhTaJqZHoYJ5y5DULsspd09RAyB8MEnpvxtbjiY"
        url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=csv'
    
    elif subset == "challenge":
        gsheetkey="1LbZcatFmM1Xse_Ou99LWE-Db02yuhZwczkM0dCk3KSk"
        url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=csv'
    else:
        raise ValueError("not supported")

    ai2arc = pd.read_csv(url)
    df = ai2arc.dropna()

    for col in ("trueStatement", "falseStatement"):
        df.loc[:,col] = df[col].apply(str.capitalize).apply(lambda x: x[0].lower() + x[1:])
        df.loc[:,col] = df[col].apply(str.strip)
        df.loc[:,col] = df[col].apply(lambda s: s if s.endswith(".") else s + ".")
        df.loc[:,col] = df[col].apply(lambda s: s.replace("..", ".") if s.endswith("..") else s)
    return [(true, false) for true, false in zip(df.trueStatement.values.tolist(), df.falseStatement.values.tolist())]





In [8]:
PAIRED_EASY_STATEMENTS = get_ai2arc_dataset_pairs("easy")
PAIRED_CHALLENGE_STATEMENTS = get_ai2arc_dataset_pairs("challenge")

len(PAIRED_EASY_STATEMENTS) + len(PAIRED_CHALLENGE_STATEMENTS)

203

In [9]:
AI2ARC_MODELS = sorted(all_models_top_preds_ai2arc["model"].unique())

#### Easy AI2ARC

In [10]:
ai2arc_easy, ai2arc_easy_df = run_paired_rest(all_models_top_preds_ai2arc, models=AI2ARC_MODELS, paired_sents=PAIRED_EASY_STATEMENTS)
ai2arc_easy.head()

Processing allenai/OLMo-7B-Instruct__sampling-based
Processing google/gemma-1.1-2b-it__sampling-based
Processing gpt-3.5-turbo-0125__top-k
Processing gpt-4-turbo-2024-04-09__top-k
Processing gpt-4o-2024-05-13__top-k
Processing meta-llama/Llama-3-70b-chat-hf__sampling-based
-> Skipped expression uncertain
Processing mistralai/Mixtral-8x22B-Instruct-v0.1__sampling-based
-> Skipped expression uncertain
Processing models/gemini-pro__sampling-based


,wilcoxon statistic,wilcoxon pval,wilcoxon,model,expr
0,272.0,0.000021,${272.0}_{2.09e-05}$,allenai/OLMo-7B-Instruct__sampling-based,almost certain
1,348.0,0.000061,${348.0}_{6.13e-05}$,allenai/OLMo-7B-Instruct__sampling-based,highly likely
2,400.0,0.000269,${400.0}_{2.69e-04}$,allenai/OLMo-7B-Instruct__sampling-based,very likely
3,622.0,0.000582,${622.0}_{5.82e-04}$,allenai/OLMo-7B-Instruct__sampling-based,probable
4,1003.5,0.046466,${1003.5}_{4.65e-02}$,allenai/OLMo-7B-Instruct__sampling-based,somewhat likely


In [11]:
print(ai2arc_easy_df.dropna().drop(["index"], axis=1).set_index("model").to_latex())

\begin{tabular}{llll}
\toprule
 & methodology & Statistic & p-value \\
model &  &  &  \\
\midrule
\olmo & sampling & 111281.00 & \textbf{<0.0001*} \\
\gemma & sampling & 46702.50 & 0.1148 \\
\chatgpt & top-k & 183290.50 & \textbf{<0.0001*} \\
\gptf & top-k & 303749.50 & \textbf{<0.0001*} \\
\gptfo & top-k & 346533.00 & \textbf{<0.0001*} \\
\llama & sampling & 250250.00 & \textbf{<0.0001*} \\
\mixtralmoelg & sampling & 129417.50 & \textbf{<0.0001*} \\
\gemini & sampling & 180349.50 & \textbf{<0.0001*} \\
\bottomrule
\end{tabular}



#### Challenge AI2ARC

In [12]:
ai2arc_chall, ai2arc_chall_df = run_paired_rest(all_models_top_preds_ai2arc, models=AI2ARC_MODELS, paired_sents=PAIRED_CHALLENGE_STATEMENTS)
ai2arc_chall

Processing allenai/OLMo-7B-Instruct__sampling-based
Processing google/gemma-1.1-2b-it__sampling-based
Processing gpt-3.5-turbo-0125__top-k
Processing gpt-4-turbo-2024-04-09__top-k
Processing gpt-4o-2024-05-13__top-k
Processing meta-llama/Llama-3-70b-chat-hf__sampling-based
Processing mistralai/Mixtral-8x22B-Instruct-v0.1__sampling-based
-> Skipped expression very unlikely
-> Skipped expression highly unlikely
Processing models/gemini-pro__sampling-based
-> Skipped expression uncertain


,wilcoxon statistic,wilcoxon pval,wilcoxon,model,expr
0,37.0,0.358553,${37.0}_{3.59e-01}$,allenai/OLMo-7B-Instruct__sampling-based,almost certain
1,46.5,0.471565,${46.5}_{4.72e-01}$,allenai/OLMo-7B-Instruct__sampling-based,highly likely
2,68.5,0.489591,${68.5}_{4.90e-01}$,allenai/OLMo-7B-Instruct__sampling-based,very likely
3,265.0,0.076989,${265.0}_{7.70e-02}$,allenai/OLMo-7B-Instruct__sampling-based,probable
4,529.5,0.163875,${529.5}_{1.64e-01}$,allenai/OLMo-7B-Instruct__sampling-based,somewhat likely
...,...,...,...,...,...
99,390.0,0.000005,${390.0}_{5.39e-06}$,models/gemini-pro__sampling-based,unlikely
100,78.0,0.000661,${78.0}_{6.61e-04}$,models/gemini-pro__sampling-based,not likely
101,171.0,0.000040,${171.0}_{4.04e-05}$,models/gemini-pro__sampling-based,doubtful
102,67.0,0.009368,${67.0}_{9.37e-03}$,models/gemini-pro__sampling-based,very unlikely


In [13]:
print(ai2arc_chall_df.dropna().drop(["index"], axis=1).set_index("model").to_latex())

\begin{tabular}{llll}
\toprule
 & methodology & Statistic & p-value \\
model &  &  &  \\
\midrule
\olmo & sampling & 48763.50 & 0.0333 \\
\gemma & sampling & 34161.50 & 0.0191 \\
\chatgpt & top-k & 123271.00 & \textbf{<0.0001*} \\
\gptf & top-k & 194138.00 & \textbf{<0.0001*} \\
\gptfo & top-k & 231543.50 & \textbf{<0.0001*} \\
\llama & sampling & 135453.00 & \textbf{<0.0001*} \\
\mixtralmoelg & sampling & 99689.50 & \textbf{<0.0001*} \\
\gemini & sampling & 106837.00 & \textbf{<0.0001*} \\
\bottomrule
\end{tabular}

